# Cuaderno U2-05. Buenas prácticas en el uso de herramientas computacionales

**Modelación y Simulación Computacional** · Maestría en Ingeniería, Universidad de Sucre, periodo 2026-2
**Unidad 2.** Herramientas computacionales para modelación y simulación
**Subtema del plan.** 2.5 Buenas prácticas en el uso de herramientas computacionales
**Autor.** Prof. Daniel Otero Meza, Ing., Ph.D.

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad2/U2_05_buenas_practicas.ipynb)


Este cuaderno recorre la Sección 2.6 del libro. Escribe funciones
puras, declara el dominio físico del modelo con excepciones y aserciones,
implementa las tres pruebas mínimas de todo modelo y aplica el protocolo del
Algoritmo 2.3 a un fragmento sugerido por un asistente automático, con el
diagnóstico completo del Ejemplo 2.7.

## Objetivos de aprendizaje

Al terminar este cuaderno el estudiante debe ser capaz de lo siguiente.

1. Distinguir una función pura de una con efecto lateral y reescribir la trampa del argumento mutable por omisión.
2. Declarar el dominio de validez de una función del modelo con excepciones y aserciones, como en el Listado 2.14.
3. Escribir las tres pruebas mínimas de un modelo, que son el caso límite analítico, el balance global y la invariancia de unidades.
4. Reconocer un error silencioso según la Definición 2.14 y explicar por qué ninguna revisión superficial lo detecta.
5. Aplicar el protocolo del Algoritmo 2.3 a un fragmento asistido y reproducir las cifras del Ejemplo 2.7.

## Puesta a punto

La primera celda instala lo que falte y la segunda fija la semilla del curso,
la paleta del libro y la función que compara cada resultado con el valor
publicado. Ningún resultado de este cuaderno depende de una ejecución
concreta.

In [ ]:
# Puesta a punto. Detecta el entorno e instala solo lo que falte.
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict[str, str]) -> None:
    """Instala los paquetes cuyo módulo no se encuentre en el entorno."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        *faltantes], check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("entorno listo, Colab =", EN_COLAB)

In [ ]:
# Configuración común a todos los cuadernos del curso.
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sympy as sp

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

PALETA = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
          "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 110,
                     "font.size": 9, "axes.grid": True,
                     "grid.linewidth": 0.4, "grid.alpha": 0.5,
                     "axes.prop_cycle": plt.cycler(color=list(PALETA.values()))})


def contra_libro(nombre: str, calculado: float, publicado: float,
                 unidad: str = "", tol: float = 1e-3, relativa: bool = True,
                 exigir: bool = True, nota: str = "") -> None:
    """Compara un resultado del cuaderno con el valor que publica el libro.

    Detiene la ejecución si la diferencia excede la tolerancia y `exigir` es
    verdadero. Las magnitudes que dependen de la máquina, como los tiempos de
    ejecución, se informan con `exigir=False` y una nota que lo advierte.
    """
    error = abs(calculado - publicado)
    if relativa and publicado != 0.0:
        error = error / abs(publicado)
    ok = error <= tol
    print(f"{nombre:<44s} cuaderno {calculado:>13.6g}  "
          f"libro {publicado:>13.6g} {unidad:<9s} "
          f"{'coincide' if ok else 'DIFIERE '}{nota}")
    if exigir and not ok:
        raise AssertionError(
            f"{nombre}, el cuaderno da {calculado!r} y el libro publica "
            f"{publicado!r}, con error {error:.3e}")


print("NumPy", np.__version__, "| SciPy", scipy.__version__,
      "| pandas", pd.__version__, "| SymPy", sp.__version__)

### Acceso a los datos

Los archivos viven en `03_cuadernos/datos/`. La función `ruta_datos` intenta
primero la ruta relativa del repositorio y, si el archivo no está, lo regenera
con la semilla del curso. Así el cuaderno corre igual en Colab, donde no
existe la carpeta, y en una instalación local. Nunca se usan rutas absolutas
del computador del docente.

In [ ]:
# Acceso a los datos. Se intenta la ruta relativa del repositorio y, si el
# archivo no existe, se regenera con la semilla del curso.
IRRADIANCIA = (
    42.9, 68.8, 100.0, 140.0, 176.8, 221.8, 274.8, 319.9, 369.7, 418.8,
    478.3, 543.3, 591.9, 647.3, 686.9, 732.8, 767.3, 825.9, 847.7, 881.1,
    911.0, 940.3, 934.4, 942.8, 962.3, 943.3, 938.4, 939.6, 932.1, 875.1,
    860.8, 820.8, 789.0, 745.4, 675.3, 634.7, 581.3, 542.0, 488.0, 424.1,
    376.2, 318.6, 264.8, 220.1, 176.2, 137.0, 98.0, 69.4, 43.3,
)


def _gen_irradiancia(destino: Path) -> None:
    """Registro del piranómetro del Ejemplo 2.7, 49 muestras completas."""
    marca = pd.date_range("2026-03-04 06:00", periods=49, freq="15min")
    pd.DataFrame({"fecha_hora": marca.strftime("%Y-%m-%d %H:%M:%S"),
                  "irradiancia_W_m2": IRRADIANCIA}).to_csv(
        destino, index=False, encoding="utf-8")


GENERADORES = {
    "irradiancia_piranometro.csv": _gen_irradiancia,
}

CANDIDATAS = [Path("datos"), Path("..") / "datos",
              Path("03_cuadernos") / "datos", Path("..") / ".." / "datos"]


def ruta_datos(nombre: str) -> Path:
    """Devuelve la ruta del archivo de datos, generándolo si hace falta."""
    for base in CANDIDATAS:
        candidata = base / nombre
        if candidata.is_file():
            return candidata
    generada = Path("salida") / "datos_generados"
    generada.mkdir(parents=True, exist_ok=True)
    destino = generada / nombre
    if not destino.is_file():
        GENERADORES[nombre](destino)
    return destino

print("datos disponibles en", ruta_datos("irradiancia_piranometro.csv").parent)

## 1. Funciones puras y la trampa del argumento
mutable

La Definición 2.13 del libro define la función pura como aquella cuyo valor de
retorno depende únicamente de sus argumentos y que no modifica ningún estado
externo. El incumplimiento más común en Python es también el más silencioso y
proviene de usar un objeto mutable como valor por omisión. El Listado 2.13
publica los valores 41.8, 42.05 y 42.4 al procesar las tres primeras lecturas
de la estación de bombeo, y luego 34.3 en lugar de 10.0 al comenzar una serie
nueva.

In [ ]:
# Listado 2.13 del libro.
def promedio_acumulado(q, historial=[]):    # incorrecto
    """La lista por omisión se crea una sola vez y sobrevive."""
    historial.append(q)
    return sum(historial) / len(historial)


def promedio_puro(historial, q):
    """Devuelve el nuevo historial y su media, sin alterar el original."""
    nuevo = (*historial, q)
    return nuevo, sum(nuevo) / len(nuevo)


LECTURAS = (41.8, 42.3, 43.1)      # L/s, tres primeras de la estación
obtenidos = [promedio_acumulado(q) for q in LECTURAS]
for q, valor, esperado in zip(LECTURAS, obtenidos, (41.8, 42.05, 42.4)):
    contra_libro(f"promedio acumulado tras {q} L/s", valor, esperado, "L/s",
                 tol=1e-9)

serie_nueva = promedio_acumulado(10.0)
contra_libro("serie nueva con una sola lectura de 10.0", serie_nueva, 34.3,
             "L/s", tol=1e-9)
print("\nel valor correcto para una serie nueva con una sola lectura es 10.0, "
      "y la función devuelve 34.3 porque la lista por omisión se creó al "
      "definirla y conserva las lecturas anteriores")

In [ ]:
historial = ()
for q in LECTURAS:
    historial, media = promedio_puro(historial, q)
    print(f"lectura {q:5.1f} L/s, media acumulada {media:7.4f} L/s")

historial_nuevo, media_nueva = promedio_puro((), 10.0)
contra_libro("serie nueva con la versión pura", media_nueva, 10.0, "L/s",
             tol=1e-9)
assert historial == (41.8, 42.3, 43.1), "el historial original no debe alterarse"
print("\ndos llamadas con los mismos argumentos devuelven siempre lo mismo, "
      "de modo que la función puede probarse en aislamiento y sustituirse por "
      "su resultado sin alterar el programa")

## 2. El dominio de validez como contrato

La excepción se lanza ante una entrada externa inválida y el usuario debe
poder manejarla, mientras que la aserción comprueba una condición que el
propio código garantiza y cuyo incumplimiento indica un defecto de
programación. El Listado 2.14 del libro aplica los dos sobre la ecuación de
Manning, y publica un caudal de 4.8275 m3/s con rugosidad de 0.014, área de
2.40 m2, radio hidráulico de 0.62 m y pendiente de 0.0015.

In [ ]:
# Listado 2.14 del libro.
def caudal_manning(n, A, R_h, S_0):
    """Caudal por la ecuación de Manning, en m3/s."""
    if not (0.008 <= n <= 0.20):
        raise ValueError(f"rugosidad fuera de rango: n = {n}")
    if A <= 0.0 or R_h <= 0.0:
        raise ValueError("el área y el radio hidráulico deben ser positivos")
    if not (1e-6 <= S_0 <= 0.20):
        raise ValueError(f"pendiente fuera de rango: S_0 = {S_0}")
    Q = A * R_h ** (2.0 / 3.0) * np.sqrt(S_0) / n
    assert np.isfinite(Q) and Q > 0.0, "caudal no físico"
    return Q


Q_manning = caudal_manning(0.014, 2.40, 0.62, 0.0015)
contra_libro("caudal de Manning", Q_manning, 4.8275, "m3/s", tol=5e-5,
             relativa=False)

for entrada in [(0.0, 2.40, 0.62, 0.0015), (0.014, -1.0, 0.62, 0.0015),
                (0.014, 2.40, 0.62, -0.002)]:
    try:
        caudal_manning(*entrada)
    except ValueError as error:
        print(f"  {str(entrada):<32s} -> ValueError, {error}")
    else:
        raise AssertionError(f"la entrada {entrada} debió rechazarse")
print("\ncon rugosidad nula la función se niega a calcular y nombra el valor "
      "ofensivo, en lugar de devolver un infinito que contaminaría todo el "
      "cálculo aguas abajo")

## 3. Las tres pruebas mínimas de un modelo

El libro sostiene que las pruebas de un modelo no comprueban que una función
devuelva un valor previamente anotado, sino que el modelo respete una ley
física. Tres pruebas bastan para atrapar la mayoría de los errores
estructurales, a saber, la comparación con un caso límite de solución
analítica, el cierre de un balance de masa o de energía y la invariancia del
resultado ante un cambio coherente de unidades. El Listado 2.15 implementa la
segunda sobre una cadena de dos tanques, y el libro publica una deriva de masa
de 4.263e-14 kg sobre un total de 120 kg.

In [ ]:
# Listado 2.15 del libro.
from scipy.integrate import solve_ivp


def tanques(t, y, k):
    """Dos tanques en serie sin fuentes ni sumideros, en kg."""
    return np.array([-k * y[0], k * y[0] - k * y[1], k * y[1]])


def test_conservacion_masa():
    y0 = np.array([120.0, 0.0, 0.0])          # kg
    sol = solve_ivp(tanques, (0.0, 40.0), y0, args=(0.18,),
                    rtol=1e-9, atol=1e-12)
    deriva = np.max(np.abs(sol.y.sum(axis=0) - y0.sum()))
    assert deriva < 1e-8 * y0.sum(), f"fuga de masa de {deriva:.3e} kg"
    return deriva


deriva = test_conservacion_masa()
contra_libro("deriva de masa", deriva, 4.263e-14, "kg", tol=0.5,
             exigir=False,
             nota="  (del orden de la precisión de la máquina)")
print(f"\nla deriva es {deriva / 120.0:.2e} veces la masa total, del orden de "
      "la precisión de la máquina")

In [ ]:
def test_caso_limite():
    """Con k igual en los dos tanques, el primero decae exponencialmente."""
    k = 0.18
    sol = solve_ivp(tanques, (0.0, 40.0), np.array([120.0, 0.0, 0.0]),
                    args=(k,), rtol=1e-10, atol=1e-13, dense_output=True)
    malla = np.linspace(0.0, 40.0, 200)
    exacta = 120.0 * np.exp(-k * malla)
    error = float(np.abs(sol.sol(malla)[0] - exacta).max())
    assert error < 1e-6, f"el primer tanque no sigue la solución exacta, {error:.2e}"
    return error


def test_invariancia_unidades():
    """El resultado no puede depender de trabajar en kg o en gramos."""
    k = 0.18
    en_kg = solve_ivp(tanques, (0.0, 40.0), np.array([120.0, 0.0, 0.0]),
                      args=(k,), rtol=1e-10, atol=1e-13).y[:, -1]
    en_g = solve_ivp(tanques, (0.0, 40.0), np.array([120.0e3, 0.0, 0.0]),
                     args=(k,), rtol=1e-10, atol=1e-13).y[:, -1]
    error = float(np.abs(en_g / 1000.0 - en_kg).max())
    assert error < 1e-6, f"el modelo no es invariante ante el cambio de unidad"
    return error


print(f"caso límite analítico, error máximo   {test_caso_limite():.3e} kg")
print(f"invariancia de unidades, error máximo {test_invariancia_unidades():.3e} kg")
print(f"balance global, deriva                {deriva:.3e} kg")

In [ ]:
# Si alguien cambia un signo, la prueba falla de inmediato y cuantifica la fuga.
def tanques_con_error(t, y, k):
    """Versión con un signo cambiado en la segunda ecuación."""
    return np.array([-k * y[0], k * y[0] + k * y[1], k * y[1]])


sol_mala = solve_ivp(tanques_con_error, (0.0, 40.0),
                     np.array([120.0, 0.0, 0.0]), args=(0.18,),
                     rtol=1e-9, atol=1e-12)
fuga = float(np.max(np.abs(sol_mala.y.sum(axis=0) - 120.0)))
print(f"con el signo cambiado la fuga de masa asciende a {fuga:.3e} kg, "
      f"esto es {fuga / 120.0:.1f} veces la masa inicial")
assert fuga > 1e-8 * 120.0, "la prueba debe atrapar el signo cambiado"
print("la prueba no comprueba un número anotado, comprueba una ley física, "
      "y por eso atrapa un error que ninguna revisión de sintaxis detecta")

La Tabla 2.6 del libro generaliza estas comprobaciones
en un catálogo de siete entradas. Las tres primeras deben quedar escritas como
pruebas ejecutables dentro del repositorio del proyecto.

In [ ]:
CATALOGO = pd.DataFrame(
    [("Coherencia dimensional", "unidades de cada término",
      "factor de conversión omitido"),
     ("Caso límite analítico", "coincidencia con la solución exacta",
      "signo cambiado o índice desplazado"),
     ("Balance global", "conservación de masa o energía",
      "fuga por condición de frontera"),
     ("Dominio de validez", "rechazo de entradas imposibles",
      "extrapolación silenciosa"),
     ("Invariancia de unidades", "mismo resultado en otro sistema",
      "constante empírica mal aplicada"),
     ("Refinamiento", "independencia de la malla y del paso",
      "error de discretización"),
     ("Orden de magnitud", "escala esperada del proceso",
      "error de un factor de mil")],
    columns=["Verificación", "Qué se comprueba", "Error que atrapa"])
print(CATALOGO.to_string(index=False))

## 4. El error silencioso

La Definición 2.14 del libro describe el error silencioso como aquel que no
interrumpe la ejecución ni produce mensaje alguno, y cuyo resultado es un
número del orden de magnitud esperado, de modo que solo puede detectarse
mediante una comprobación independiente del propio cálculo. La única defensa
es un protocolo aplicado siempre y no solo cuando algo parece raro, ya que por
definición nada parecerá raro. La Figura 2.13 recoge ese protocolo y el
Algoritmo 2.3 lo expresa como procedimiento.

In [ ]:
def protocolo_verificacion(fragmento, origen, caso_limite, exacta,
                           tolerancia, entradas_invalidas):
    """Algoritmo 2.3 del libro, decide si incorporar o corregir."""
    registro = {"origen declarado": origen}

    obtenido = fragmento(*caso_limite)
    registro["error en el caso límite"] = float(abs(obtenido - exacta))
    registro["caso límite"] = registro["error en el caso límite"] <= tolerancia

    rechaza = True
    for entrada in entradas_invalidas:
        try:
            fragmento(*entrada)
        except Exception:
            continue
        rechaza = False
    registro["rechaza entradas fuera del dominio"] = rechaza

    registro["decisión"] = ("incorporar"
                            if registro["caso límite"] and rechaza
                            else "corregir")
    return pd.Series(registro)


informe_manning = protocolo_verificacion(
    caudal_manning, origen="escrito a mano",
    caso_limite=(0.014, 2.40, 0.62, 0.0015), exacta=4.8275,
    tolerancia=5e-4,
    entradas_invalidas=[(0.0, 2.40, 0.62, 0.0015),
                        (0.014, -1.0, 0.62, 0.0015),
                        (0.014, 2.40, 0.62, -0.002)])
print(informe_manning.to_string())
assert informe_manning["decisión"] == "incorporar"

## 5. Ejemplo 2.7, diagnóstico de un fragmento asistido

Un piranómetro registra la irradiancia global sobre el plano de un arreglo
fotovoltaico cada 15 min entre las 06 h 00 y las 18 h 00, para un total de 49
muestras. Una falla del registrador cerca del mediodía elimina tres muestras
consecutivas. Un asistente automático propone una función que estima la
energía generada con área de módulos de 34 m2 y rendimiento global de 0.156.
El libro publica 35.374 kWh con la serie completa, 31.597 kWh con el fragmento
asistido sobre las 46 sobrevivientes y 35.322 kWh al pasar la marca de tiempo
real como abscisa.

In [ ]:
registro = pd.read_csv(ruta_datos("irradiancia_piranometro.csv"),
                       parse_dates=["fecha_hora"])
G = registro["irradiancia_W_m2"].to_numpy()
segundos = (registro["fecha_hora"] - registro["fecha_hora"].iloc[0]
            ).dt.total_seconds().to_numpy()

A_MODULOS, ETA, P_NOMINAL = 34.0, 0.156, 6.80      # m2, adimensional, kW

print(f"muestras {G.size}, de {registro['fecha_hora'].iloc[0].time()} a "
      f"{registro['fecha_hora'].iloc[-1].time()}")
contra_libro("irradiancia máxima registrada", G.max(), 962.3, "W/m2",
             tol=0.05, relativa=False)
print(f"duración del registro {segundos[-1] / 3600.0:.2f} h, "
      f"paso nominal {np.diff(segundos).min():.0f} s")

In [ ]:
def energia_diaria_asistida(irradiancia):
    """Fragmento propuesto por el asistente, supone muestreo uniforme."""
    return A_MODULOS * ETA * np.trapezoid(irradiancia, dx=900.0) / 3.6e6


def energia_diaria_con_tiempo(irradiancia, t_segundos):
    """Versión que recibe explícitamente la marca de tiempo."""
    return (A_MODULOS * ETA
            * np.trapezoid(irradiancia, x=t_segundos) / 3.6e6)


# La falla elimina tres muestras consecutivas cerca del mediodía.
perdidas = [23, 24, 25]
sobreviven = np.array([i for i in range(G.size) if i not in perdidas])

E_completa = energia_diaria_asistida(G)
E_asistida = energia_diaria_asistida(G[sobreviven])
E_con_tiempo = energia_diaria_con_tiempo(G[sobreviven], segundos[sobreviven])

contra_libro("energía con la serie completa", E_completa, 35.374, "kWh",
             tol=5e-4, relativa=False)
contra_libro("energía del fragmento asistido", E_asistida, 31.597, "kWh",
             tol=5e-4, relativa=False)
contra_libro("energía con la marca de tiempo real", E_con_tiempo, 35.322,
             "kWh", tol=5e-4, relativa=False)

In [ ]:
irradiacion = E_completa / (A_MODULOS * ETA)
contra_libro("irradiación diaria", irradiacion, 6.669, "kWh/m2", tol=5e-4,
             relativa=False)
contra_libro("rendimiento específico verdadero", E_completa / P_NOMINAL,
             5.20, "kWh/kW", tol=0.005, relativa=False)
contra_libro("rendimiento específico del fragmento", E_asistida / P_NOMINAL,
             4.65, "kWh/kW", tol=0.005, relativa=False)
contra_libro("desviación del fragmento asistido",
             100.0 * (E_completa - E_asistida) / E_completa, 10.7, "%",
             tol=0.05, relativa=False)
contra_libro("desviación con la marca de tiempo",
             100.0 * (E_completa - E_con_tiempo) / E_completa, 0.15, "%",
             tol=0.005, relativa=False)

La verificación del Ejemplo 2.7 es la que delata el
fragmento. El asistente supone implícitamente una duración de 45 intervalos de
900 s, esto es 11.25 h, mientras que el registro abarca 12.00 h. Esa
discrepancia de tres cuartos de hora es la comprobación independiente que el
protocolo exige, porque el rendimiento específico que produce el fragmento es
plausible para la región y por sí solo no habría levantado sospecha.

In [ ]:
duracion_supuesta = (sobreviven.size - 1) * 900.0 / 3600.0
duracion_real = (segundos[-1] - segundos[0]) / 3600.0

contra_libro("duración que supone el fragmento", duracion_supuesta, 11.25,
             "h", tol=0.005, relativa=False)
contra_libro("duración real del registro", duracion_real, 12.00, "h",
             tol=0.005, relativa=False)
print(f"\nla diferencia es de {60 * (duracion_real - duracion_supuesta):.0f} "
      "min, exactamente el hueco que dejó la falla")
print("\nel fragmento no contiene ningún error de programación. La función "
      "existe, los argumentos son válidos y el resultado tiene las unidades "
      "correctas. El error está en una hipótesis no declarada, la del "
      "muestreo uniforme, que era cierta cuando el asistente escribió el "
      "código y dejó de serlo cuando el registrador falló")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13.5 / 2.54, 5.6 / 2.54),
                         layout="constrained")
ax = ejes[0]
ax.plot(segundos / 3600.0 + 6.0, G, "-", color=PALETA["gris"], lw=1.0,
        label="serie completa")
ax.plot(segundos[sobreviven] / 3600.0 + 6.0, G[sobreviven], "o", ms=3.0,
        color=PALETA["azul"], label="muestras sobrevivientes")
ax.plot(segundos[perdidas] / 3600.0 + 6.0, G[perdidas], "x", ms=6.0,
        color=PALETA["rojo"], label="muestras perdidas")
ax.set_xlabel("Hora del día (h)")
ax.set_ylabel("Irradiancia sobre el plano (W/m2)")
ax.set_xlim(6.0, 18.0)
ax.set_ylim(0.0, 1050.0)
ax.legend(loc="lower center", fontsize=7.0)

ax = ejes[1]
etiquetas = ["serie\ncompleta", "fragmento\nasistido", "con marca\nde tiempo"]
valores = [E_completa, E_asistida, E_con_tiempo]
ax.bar(etiquetas, valores,
       color=[PALETA["gris"], PALETA["rojo"], PALETA["verde"]], width=0.55)
ax.axhline(E_completa, color=PALETA["gris"], lw=0.8, ls="--")
ax.set_ylabel("Energía diaria (kWh)")
ax.set_ylim(0.0, 40.0)
for i, v in enumerate(valores):
    ax.text(i, v + 0.8, f"{v:.3f}", ha="center", fontsize=7.5)
plt.show()

## 6. El protocolo de declaración

El protocolo que exige la asignatura consta de tres elementos y no admite
excepciones. El módulo indica en su encabezado qué partes fueron sugeridas por
un asistente y cuáles se escribieron a mano, cada fragmento de origen asistido
lleva asociada al menos una prueba ejecutable que comprueba una propiedad
física del resultado y no simplemente su valor, y el informe final declara el
uso de la herramienta junto con los supuestos y las limitaciones del modelo.
Esa declaración permite a quien audite el trabajo concentrar la revisión donde
el riesgo es mayor.

In [ ]:
CABECERA = [
    "Estimación de energía de un arreglo fotovoltaico.",
    "",
    "Origen del código",
    "-----------------",
    "- energia_diaria_con_tiempo, sugerida por un asistente automático y",
    "  corregida a mano para recibir la marca de tiempo. Prueba asociada,",
    "  test_energia_invariante_ante_huecos.",
    "- El resto se escribió a mano.",
    "",
    "Supuestos",
    "---------",
    "El rendimiento global de 0.156 incluye el módulo y las pérdidas del",
    "sistema. La irradiancia se integra por la regla del trapecio sobre la",
    "marca de tiempo real, de modo que el resultado no depende de que el",
    "muestreo sea uniforme.",
]

CUERPO = [
    "",
    "def energia_diaria_con_tiempo(irradiancia, t_segundos):",
    "    return A * ETA * np.trapezoid(irradiancia, x=t_segundos) / 3.6e6",
    "",
    "",
    "def test_energia_invariante_ante_huecos():",
    "    completa = energia_diaria_con_tiempo(G, segundos)",
    "    con_hueco = energia_diaria_con_tiempo(G[vivas], segundos[vivas])",
    "    assert abs(completa - con_hueco) / completa < 0.01",
]

comillas = chr(34) * 3
MODULO = Path("salida") / "modelo_fv.py"
MODULO.parent.mkdir(exist_ok=True)
MODULO.write_text(comillas + chr(10).join(CABECERA) + comillas + chr(10)
                  + chr(10).join(CUERPO) + chr(10), encoding="utf-8")
print(MODULO.read_text(encoding="utf-8"))

In [ ]:
def test_energia_invariante_ante_huecos():
    """La energía estimada no puede cambiar al perder muestras interiores."""
    completa = energia_diaria_con_tiempo(G, segundos)
    con_hueco = energia_diaria_con_tiempo(G[sobreviven], segundos[sobreviven])
    desviacion = abs(completa - con_hueco) / completa
    assert desviacion < 0.01, f"el hueco altera el resultado en {desviacion:.3%}"
    return desviacion


def test_energia_asistida_falla_con_huecos():
    """La versión asistida sí cambia, y esa es la prueba que la delata."""
    completa = energia_diaria_asistida(G)
    con_hueco = energia_diaria_asistida(G[sobreviven])
    return abs(completa - con_hueco) / completa


print(f"versión corregida, desviación ante el hueco "
      f"{test_energia_invariante_ante_huecos():.4%}")
print(f"versión asistida, desviación ante el hueco  "
      f"{test_energia_asistida_falla_con_huecos():.4%}")
print("\nuna sola prueba de propiedad física separa las dos versiones, "
      "mientras que ninguna comprobación de valor lo habría hecho, porque "
      "los dos números son plausibles")

## 7. Ejercicios guiados

Cinco celdas incompletas con su verificación inmediatamente después.

### Ejercicio 1. Reescritura de una función impura

Reescriba como función pura una rutina que acumula el balance de un tanque en
una lista por omisión.

In [ ]:
# COMPLETE: devuelva una pareja formada por el nuevo estado acumulado y el
# balance, sin modificar el estado recibido ni guardar estado externo.
REVISAR_1 = False


def balance_impuro(entrada, salida, acumulado=[]):     # incorrecto
    acumulado.append(entrada - salida)
    return sum(acumulado)


def balance_puro(acumulado, entrada, salida):
    """Devuelve el nuevo acumulado y el balance total, en kg."""
    return acumulado, 0.0      # <- reemplace por la versión pura

In [ ]:
if REVISAR_1:
    estado = ()
    for entrada, salida in ((120.0, 100.0), (118.0, 105.0), (125.0, 121.0)):
        estado, total = balance_puro(estado, entrada, salida)
    assert estado == (20.0, 13.0, 4.0), f"acumulado inesperado {estado}"
    assert abs(total - 37.0) < 1e-12, f"balance inesperado {total}"
    reinicio, total_reinicio = balance_puro((), 10.0, 4.0)
    assert abs(total_reinicio - 6.0) < 1e-12, \
        "una serie nueva debe empezar de cero"
    previo = balance_impuro(10.0, 4.0)
    repetido = balance_impuro(10.0, 4.0)
    print(f"la versión impura devuelve {previo} y luego {repetido} con los "
          "mismos argumentos")
    assert previo != repetido, "la versión impura arrastra estado"
    print("ejercicio 1 correcto, la versión pura no arrastra estado")
else:
    print("ejercicio 1 pendiente, complete la celda y ponga REVISAR_1 = True")

### Ejercicio 2. Aserciones de dominio físico

Escriba la función que calcula el número de Reynolds de una tubería y declara
su dominio de validez con excepciones para las entradas externas y una
aserción para lo que el propio código garantiza.

In [ ]:
# COMPLETE: lance ValueError si el diámetro, la velocidad o la viscosidad no
# son positivos, y afirme con assert que el resultado es finito y positivo.
REVISAR_2 = False


def reynolds(rho, v, D, mu):
    """Número de Reynolds, adimensional. rho en kg/m3, v en m/s, D en m."""
    return rho * v * D / mu    # <- agregue el contrato antes y después

In [ ]:
if REVISAR_2:
    Re = reynolds(998.0, 1.20, 0.100, 1.002e-3)
    print(f"número de Reynolds {Re:.4e}, régimen turbulento" if Re > 4000
          else f"número de Reynolds {Re:.4e}")
    assert abs(Re - 998.0 * 1.20 * 0.100 / 1.002e-3) < 1e-6
    for entrada in [(-1.0, 1.2, 0.1, 1e-3), (998.0, 0.0, 0.1, 1e-3),
                    (998.0, 1.2, -0.1, 1e-3), (998.0, 1.2, 0.1, 0.0)]:
        try:
            reynolds(*entrada)
        except ValueError:
            continue
        raise AssertionError(f"la entrada {entrada} debió rechazarse")
    print("ejercicio 2 correcto, el contrato rechaza las cuatro entradas "
          "imposibles")
else:
    print("ejercicio 2 pendiente, complete la celda y ponga REVISAR_2 = True")

### Ejercicio 3. Problema 2-24, módulo de un digestor
anaerobio

Escriba los parámetros de un digestor anaerobio como estructura inmutable, la
derivada del modelo y tres pruebas de caso límite, balance de masa e
invariancia de unidades. Este es el Problema 2-24 del libro.

In [ ]:
# COMPLETE: escriba la derivada del digestor, con y = (S, X) en kg de DQO por
# metro cúbico y de biomasa por metro cúbico. La cinética es de Monod y el
# reactor es completamente mezclado con dilución D.
REVISAR_3 = False
from dataclasses import dataclass


@dataclass(frozen=True)
class Digestor:
    mu_max: float = 0.30        # 1/d
    K_s: float = 0.85           # kg DQO/m3
    Y: float = 0.08             # kg biomasa por kg DQO
    kd: float = 0.02            # 1/d
    D: float = 0.10             # 1/d
    S_in: float = 12.0          # kg DQO/m3


def derivada_digestor(t, y, p):
    """Devuelve dy/dt del digestor, con y = (S, X)."""
    return np.zeros(2)          # <- reemplace por el modelo

In [ ]:
if REVISAR_3:
    dig = Digestor()

    def test_limite_sin_biomasa():
        """Sin biomasa el sustrato tiende al de alimentación."""
        sol = solve_ivp(derivada_digestor, (0.0, 200.0),
                        np.array([2.0, 0.0]), args=(dig,), rtol=1e-9,
                        atol=1e-12)
        exacta = dig.S_in + (2.0 - dig.S_in) * np.exp(-dig.D * 200.0)
        error = abs(float(sol.y[0, -1]) - exacta)
        assert error < 1e-6, f"el caso límite falla por {error:.2e}"
        return error

    def test_balance_dqo():
        """Con Y igual a uno y decaimiento nulo la DQO total se conserva."""
        cerrado = Digestor(Y=1.0, kd=0.0, D=0.0, S_in=0.0)
        sol = solve_ivp(derivada_digestor, (0.0, 60.0),
                        np.array([8.0, 0.5]), args=(cerrado,), rtol=1e-10,
                        atol=1e-13)
        deriva = float(np.max(np.abs(sol.y.sum(axis=0) - 8.5)))
        assert deriva < 1e-8 * 8.5, f"fuga de DQO de {deriva:.3e}"
        return deriva

    def test_invariancia_unidades():
        """Pasar de kg/m3 a g/m3 no puede cambiar el estado estacionario."""
        en_kg = solve_ivp(derivada_digestor, (0.0, 400.0),
                          np.array([2.0, 0.5]), args=(dig,), rtol=1e-10,
                          atol=1e-13).y[:, -1]
        en_g = solve_ivp(derivada_digestor, (0.0, 400.0),
                         np.array([2.0e3, 0.5e3]),
                         args=(Digestor(K_s=dig.K_s * 1e3,
                                        S_in=dig.S_in * 1e3),),
                         rtol=1e-10, atol=1e-10).y[:, -1]
        error = float(np.abs(en_g / 1e3 - en_kg).max())
        assert error < 1e-5, f"el modelo no es invariante, error {error:.2e}"
        return error

    print(f"caso límite sin biomasa, error   {test_limite_sin_biomasa():.3e}")
    print(f"balance de DQO, deriva           {test_balance_dqo():.3e}")
    print(f"invariancia de unidades, error   {test_invariancia_unidades():.3e}")
    print("\nejercicio 3 correcto, las tres pruebas mínimas pasan")
else:
    print("ejercicio 3 pendiente, complete la celda y ponga REVISAR_3 = True")

### Ejercicio 4. Problema 2-27, aplicación del
protocolo

Aplique el Algoritmo 2.3 a un fragmento ajeno y redacte el informe con las
anomalías, la corrección propuesta y las pruebas agregadas. Este es el
Problema 2-27 del libro. El fragmento por revisar calcula el caudal medio de
una serie con huecos.

In [ ]:
# COMPLETE: escriba la versión corregida de `media_ajena`, que debe ignorar
# los valores ausentes en lugar de tratarlos como ceros.
REVISAR_4 = False


def media_ajena(q):
    """Fragmento ajeno, trata los ausentes como ceros."""
    q = np.nan_to_num(np.asarray(q, dtype=float))
    return float(q.sum() / q.size)


def media_corregida(q):
    """Media que ignora los valores ausentes."""
    return 0.0        # <- reemplace por la corrección

In [ ]:
if REVISAR_4:
    serie_ej = np.array([44.0, 43.6, np.nan, 44.5, np.nan, 43.9, 44.2])
    ajena, corregida = media_ajena(serie_ej), media_corregida(serie_ej)
    print(f"fragmento ajeno   {ajena:7.4f} L/s")
    print(f"versión corregida {corregida:7.4f} L/s")
    print(f"sesgo del fragmento {100 * (ajena - corregida) / corregida:+.1f} %")
    assert abs(corregida - 44.04) < 5e-3, "la media válida es 44.04 L/s"
    assert ajena < corregida, "tratar los ausentes como ceros sesga hacia abajo"
    try:
        media_corregida([np.nan, np.nan])
    except ValueError:
        pass
    else:
        raise AssertionError("una serie sin lecturas válidas debe rechazarse")
    print("\nanomalía, el fragmento reemplaza los ausentes por ceros y "
          "reporta un caudal medio inferior al real. Corrección, promediar "
          "solo las lecturas válidas y rechazar la serie sin ninguna. "
          "Pruebas agregadas, una de sesgo con huecos y una de dominio")
    print("ejercicio 4 correcto")
else:
    print("ejercicio 4 pendiente, complete la celda y ponga REVISAR_4 = True")

### Ejercicio 5. Problema 2-30, catálogo propio

Proponga un catálogo de verificaciones para su línea de investigación, con
seis entradas que den la propiedad física comprobada, la prueba ejecutable y
la tolerancia. Este es el Problema 2-30 del libro. Aquí se completa el
catálogo para el arreglo fotovoltaico del Ejemplo 2.7 y el estudiante lo
adapta a su propio problema.

In [ ]:
# COMPLETE: agregue al catálogo las seis entradas con su propiedad física, el
# nombre de la prueba ejecutable y la tolerancia declarada.
REVISAR_5 = False
catalogo_propio = pd.DataFrame(
    columns=["propiedad", "prueba", "tolerancia"])

In [ ]:
if REVISAR_5:
    print(catalogo_propio.to_string(index=False))
    assert len(catalogo_propio) == 6, "el catálogo debe tener seis entradas"
    assert catalogo_propio["prueba"].is_unique, "cada prueba debe ser distinta"
    assert (catalogo_propio["tolerancia"] >= 0.0).all(), \
        "la tolerancia no puede ser negativa"

    # Dos de las seis, implementadas para mostrar la forma de la entrega.
    def test_energia_lineal_en_area():
        doble = 2.0 * A_MODULOS * ETA * np.trapezoid(G, x=segundos) / 3.6e6
        return abs(doble - 2.0 * E_completa)

    def test_energia_nula_sin_sol():
        return abs(energia_diaria_con_tiempo(np.zeros_like(G), segundos))

    print(f"\nproporcionalidad con el área, error {test_energia_lineal_en_area():.3e}")
    print(f"energía nocturna, valor            {test_energia_nula_sin_sol():.3e}")
    assert test_energia_lineal_en_area() < 1e-12
    assert test_energia_nula_sin_sol() < 1e-12
    print("ejercicio 5 correcto")
else:
    print("ejercicio 5 pendiente, complete la celda y ponga REVISAR_5 = True")

## 8. Problemas del capítulo

Los Problemas 2-24, 2-27 y 2-30 quedaron resueltos como Ejercicios 3, 4 y 5.
Se abordan aquí el 2-5 y el 2-29.

### Problema 2-5

Distinga entre lanzar una excepción y afirmar una aserción, e indique cuál
corresponde a una pendiente negativa recibida y cuál a un caudal infinito.

In [ ]:
DISTINCION = pd.DataFrame(
    [("Excepción", "entrada externa inválida",
      "el usuario puede manejarla y corregir el dato",
      "pendiente negativa recibida como argumento"),
     ("Aserción", "condición que el propio código garantiza",
      "su incumplimiento indica un defecto de programación",
      "caudal infinito calculado por la propia función")],
    columns=["Mecanismo", "Qué comprueba", "Quién responde", "Caso del "
             "Listado 2.14"])
print(DISTINCION.to_string(index=False))

# La pendiente negativa entra por la puerta y se rechaza con excepción.
try:
    caudal_manning(0.014, 2.40, 0.62, -0.002)
except ValueError as error:
    print(f"\nexcepción -> {error}")

# El caudal infinito sería un defecto interno, y la aserción lo declara.
print("aserción  -> assert np.isfinite(Q) and Q > 0.0, 'caudal no físico'")
print("\nla aserción puede desactivarse con la bandera de optimización del "
      "intérprete, y por eso nunca debe usarse para validar una entrada "
      "externa. La excepción no se desactiva")

### Problema 2-29

Argumente cuándo un asistente automático mejora la calidad de un modelo de
ingeniería y cuándo la degrada, con dos ejemplos de error silencioso.

El primer ejemplo es el del Ejemplo 2.7, ya desarrollado, donde el fragmento
supone un muestreo uniforme que dejó de ser cierto. El segundo lo construye la
celda siguiente, con una constante empírica aplicada en unidades distintas de
aquellas en que fue ajustada.

In [ ]:
def et_hargreaves_asistida(Ra_MJ, Tmed, Tmax, Tmin):
    """Devuelve la evapotranspiración, según el asistente, en mm/d."""
    return 0.0023 * Ra_MJ * (Tmed + 17.8) * np.sqrt(Tmax - Tmin)


def et_hargreaves_correcta(Ra_MJ, Tmed, Tmax, Tmin):
    """La constante 0.0023 exige la radiación en mm/d equivalentes."""
    Ra_mm = Ra_MJ / 2.45          # calor latente de vaporización, MJ/kg
    return 0.0023 * Ra_mm * (Tmed + 17.8) * np.sqrt(Tmax - Tmin)


Ra, Tmed_v, Tmax_v, Tmin_v = 34.0, 26.0, 32.0, 20.0    # MJ/(m2 d), grados C
asistida = et_hargreaves_asistida(Ra, Tmed_v, Tmax_v, Tmin_v)
correcta = et_hargreaves_correcta(Ra, Tmed_v, Tmax_v, Tmin_v)
print(f"versión asistida {asistida:6.3f} mm/d")
print(f"versión correcta {correcta:6.3f} mm/d")
print(f"factor de sobrestimación {asistida / correcta:.2f}")
assert abs(asistida / correcta - 2.45) < 1e-9, \
    "la discrepancia es exactamente el calor latente omitido"
print("\nla versión asistida devuelve un número del mismo orden de "
      "magnitud, alto para un cultivo de riego pero no absurdo para un día "
      "despejado, de modo que la revisión de rutina lo deja pasar. Es un "
      "error silencioso según la Definición 2.14, y la comprobación "
      "dimensional del Listado 2.11 lo atrapa de inmediato")
print("\nel asistente mejora la calidad cuando produce el andamiaje "
      "repetitivo que el ingeniero verifica, esto es la lectura de archivos, "
      "las figuras rotuladas o las pruebas de dominio, y la degrada cuando "
      "se le confía la formulación, porque produce una solución plausible "
      "que resuelve un problema ligeramente distinto del planteado. La "
      "posición de la asignatura es que el ingeniero responde por el "
      "resultado que firma, con independencia de quién haya tecleado las "
      "líneas")

## Cierre

### Lista de comprobación

Al cerrar el cuaderno el estudiante debe poder hacer lo siguiente sin
consultar la solución.

- Reconocer un argumento mutable por omisión y reescribir la función como pura.
- Declarar el dominio de validez de una función del modelo, con excepción para la entrada externa y aserción para lo interno.
- Escribir las tres pruebas mínimas de un modelo y demostrar que atrapan un signo cambiado.
- Aplicar el protocolo del Algoritmo 2.3 a un fragmento ajeno o asistido y decidir si se incorpora o se corrige.
- Declarar el origen del código, la prueba asociada a cada fragmento asistido y los supuestos del modelo.

### Qué revisar en el libro si algo no salió

- Si la función pura no salió, la Definición 2.13 y el Listado 2.13.
- Si el contrato de dominio no salió, el Listado 2.14 y el párrafo sobre excepciones y aserciones.
- Si las pruebas no salieron, el Listado 2.15 y la Tabla 2.6.
- Si el diagnóstico del fragmento asistido no salió, la Definición 2.14, la Figura 2.13, el Algoritmo 2.3 y el Ejemplo 2.7.

### Declaración del uso de asistentes de programación

Este cuaderno se preparó con apoyo de un asistente automático de programación.
Todo fragmento se sometió al protocolo del Algoritmo 2.3 del libro y cada
resultado numérico se comprueba contra la cifra publicada mediante la función
`contra_libro`. La regla de la asignatura es que el ingeniero responde por el
resultado que firma, con independencia de quién haya tecleado las líneas.